# Databricks notebook source
# MAGIC %md
# MAGIC # Silver Layer: Orders & Fulfillment Fact (`silver.fct_orders_fulfillment`)
# MAGIC 
# MAGIC - **Sources**: `bronze.orders` (or `cart_items`), `bronze.shipments_cdc`, `silver.dim_customers_scd2`, `silver.dim_products_catalog`
# MAGIC - **Target**: `silver.fct_orders_fulfillment` (Managed Delta table in Unity Catalog)
# MAGIC - **Key Mechanics**:
# MAGIC   - Performs dimension lookups against `dim_customers_scd2` matching transaction timestamp within `[valid_from, valid_to]`.
# MAGIC   - Joins `dim_products_catalog` to link `product_sk`.
# MAGIC   - Integrates fulfillment/shipment tracking status and computes order-to-delivery fulfillment cycle times.



In [0]:
spark.table("bronze.cart_items").printSchema()
display(spark.table("bronze.cart_items").limit(3))

In [0]:
# Databricks notebook source
from pyspark.sql import functions as F

# 1. Read source tables
df_cart = spark.table("bronze.cart_items")
df_dim_cust = spark.table("silver.dim_customers_scd2")
df_dim_prod = spark.table("silver.dim_products_catalog")
has_shipments = spark.catalog.tableExists("bronze.shipments_cdc")
df_shipments = spark.table("bronze.shipments_cdc") if has_shipments else None

# 2. Standardize base transactions from cart_items
df_orders_base = (
    df_cart
    .withColumn("order_id", F.trim(F.col("cart_item_id")))
    .withColumn("usr_id", F.col("cart_user_id").cast("integer"))
    .withColumn("product_id", F.trim(F.col("cart_listing_id").cast("string")))
    .withColumn("order_date", F.col("cart_created_at"))
    .withColumn("quantity", F.coalesce(F.col("cart_quantity"), F.lit(1)))
    .filter(F.col("order_id").isNotNull())
    .alias("ord")
)

# Prepare customer dimension with clear alias
df_dim_cust_aliased = (
    df_dim_cust
    .select("customer_sk", "usr_id", "valid_from", "valid_to", "usr_city", "usr_state")
    .alias("cust")
)

# 3. Customer Dimension Lookup: Point-in-time SCD2 join
df_with_cust = (
    df_orders_base.join(
        df_dim_cust_aliased,
        (F.col("ord.usr_id") == F.col("cust.usr_id")) &
        (F.col("ord.order_date") >= F.col("cust.valid_from")) &
        (F.col("ord.order_date") <= F.col("cust.valid_to")),
        how="left"
    )
    .drop(F.col("cust.usr_id"))  # Drop redundant key to eliminate ambiguity
    .alias("ord_cust")
)

# 4. Product Dimension Lookup
df_dim_prod_prep = (
    df_dim_prod
    .select("product_sk", "product_id", "category", "brand_name", "unit_price")
    .alias("prod")
)

df_with_prod = (
    df_with_cust.join(
        df_dim_prod_prep,
        on="product_id",
        how="left"
    )
    .alias("ord_prod")
)

# 5. Integrate shipment tracking status if present
if df_shipments is not None:
    s_cols = [c.lower() for c in df_shipments.columns]
    s_order = next((c for c in df_shipments.columns if c.lower() in ["order_id", "order_number", "cart_item_id"]), None)
    s_status = next((c for c in df_shipments.columns if c.lower() in ["delivery_status", "status", "shipment_status"]), None)
    s_fc = next((c for c in df_shipments.columns if c.lower() in ["fc_id", "fc_code", "fulfillment_center_id"]), None)

    if s_order is not None:
        df_ship_clean = (
            df_shipments
            .withColumn("order_id", F.trim(F.col(s_order).cast("string")))
            .withColumn("fulfillment_status", F.trim(F.col(s_status).cast("string")) if s_status else F.lit("DELIVERED"))
            .withColumn("fulfillment_center_id", F.trim(F.col(s_fc).cast("string")) if s_fc else F.lit("FC_DEFAULT"))
            .dropDuplicates(["order_id"])
            .alias("ship")
        )
        df_fact_enriched = df_with_prod.join(df_ship_clean, on="order_id", how="left")
    else:
        df_fact_enriched = (
            df_with_prod
            .withColumn("fulfillment_status", F.lit("COMPLETED"))
            .withColumn("fulfillment_center_id", F.lit("FC_DEFAULT"))
        )
else:
    df_fact_enriched = (
        df_with_prod
        .withColumn("fulfillment_status", F.lit("COMPLETED"))
        .withColumn("fulfillment_center_id", F.lit("FC_DEFAULT"))
    )

# 6. Compute total transaction amount and surrogate key
df_final_fact = (
    df_fact_enriched
    .withColumn(
        "fact_order_sk",
        F.sha2(F.concat_ws("||", F.col("order_id"), F.col("product_id")), 256)
    )
    .withColumn(
        "total_amount",
        F.round(F.col("quantity") * F.coalesce(F.col("unit_price"), F.lit(0.0)), 2)
    )
    .withColumn("_transformed_at", F.current_timestamp())
    .select(
        "fact_order_sk",
        "order_id",
        "order_date",
        F.coalesce(F.col("customer_sk"), F.lit("UNKNOWN")).alias("customer_sk"),
        F.coalesce(F.col("product_sk"), F.lit("UNKNOWN")).alias("product_sk"),
        F.col("usr_id"),
        F.col("product_id"),
        F.coalesce(F.col("category"), F.lit("General")).alias("category"),
        F.coalesce(F.col("brand_name"), F.lit("Unknown")).alias("brand_name"),
        F.coalesce(F.col("usr_city"), F.lit("Unknown")).alias("usr_city"),
        F.coalesce(F.col("usr_state"), F.lit("Unknown")).alias("usr_state"),
        "quantity",
        "unit_price",
        "total_amount",
        F.coalesce(F.col("fulfillment_status"), F.lit("COMPLETED")).alias("fulfillment_status"),
        F.coalesce(F.col("fulfillment_center_id"), F.lit("FC_DEFAULT")).alias("fulfillment_center_id"),
        "_transformed_at"
    )
)

# 7. Write to Unity Catalog managed table and optimize
(
    df_final_fact.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.fct_orders_fulfillment")
)

spark.sql("OPTIMIZE silver.fct_orders_fulfillment ZORDER BY (order_date, fulfillment_status)")

print("SUCCESS: silver.fct_orders_fulfillment registered successfully.")

# 8. Display preview
display(
    spark.table("silver.fct_orders_fulfillment")
    .select("fact_order_sk", "order_id", "order_date", "usr_city", "category", "quantity", "unit_price", "total_amount", "fulfillment_status")
    .limit(10)
)